# Two-Doer Bottleneck: Training

Two-phase curriculum:
1. **Phase 1 — Pick Object**: Doers start at goal positions and learn object selection from the Seer's message. No navigation required.
2. **Phase 2 — Full Training**: Doers start at random positions, navigate the bottleneck corridor, then select the correct item.

Phase transition triggers automatically when Phase 1 first-try success rate exceeds the threshold.

> **Self-contained**: run from top to bottom in a fresh environment — all source files are written by the cells below.

## 1. Install Dependencies

In [ ]:
!pip install -q 'jax[cuda12]' flax optax distrax chex pillow numpy wandb huggingface_hub
!pip install -q navix

## 2. Write Source Files

The cells below recreate the full module tree so this notebook has no external dependencies.

In [ ]:
import os; os.makedirs('models', exist_ok=True); open('models/__init__.py', 'w').close()

In [ ]:
import os; os.makedirs('training', exist_ok=True); open('training/__init__.py', 'w').close()

In [ ]:
import os; os.makedirs('agents', exist_ok=True); open('agents/__init__.py', 'w').close()

In [ ]:
import os; os.makedirs('eval', exist_ok=True); open('eval/__init__.py', 'w').close()

### `models/fsq.py`

In [ ]:
%%writefile models/fsq.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence

class FSQ(nn.Module):
    """
    Finite Scalar Quantization (FSQ) module.
    Projects a continuous vector into a discrete hypercube.
    """
    # A sequence of integers defining the number of levels (L) per dimension (d).
    # e.g., levels=[5, 5, 5] means d=3 dimensions, each with 5 discrete levels.
    levels: Sequence[int]

    @nn.compact
    def __call__(self, z: jnp.ndarray) -> jnp.ndarray:
        """
        Args:
            z: The continuous "thought vector" from the Seer's reasoning module.
               Expected shape: (batch_size, ..., d) where d == len(self.levels)
        Returns:
            z_ste: The quantized vector with gradients preserved via STE.
        """
        levels = jnp.asarray(self.levels, dtype=z.dtype)
        
        # 1. Bounding: Restrict the unbounded input z to the range [-1, 1].
        # We use tanh as a standard, proven method for this projection.
        z_bound = jnp.tanh(z)
        
        # 2. Scaling: Map the [-1, 1] range to the grid [0, L - 1].
        # E.g., for L=5, this maps [-1, 1] to [0, 4].
        half_width = (levels - 1) / 2.0
        z_scaled = z_bound * half_width + half_width
        
        # 3. Quantization: Snap to the nearest integer grid point.
        if self.has_rng('noise'):
            # Inject uniform noise during training to "shake" the FSQ and prevent early mode collapse
            noise = jax.random.uniform(self.make_rng('noise'), z_scaled.shape, minval=-0.2, maxval=0.2)
            z_quantized = jnp.round(z_scaled + noise)
        else:
            z_quantized = jnp.round(z_scaled)
        
        # 4. Straight-Through Estimator (STE) Trick in JAX
        # Forward pass: Evaluates to z_quantized.
        # Backward pass: jax.lax.stop_gradient blocks the gradient from the 
        # non-differentiable z_quantized, so the gradient flows directly through z_scaled.
        z_ste = z_scaled + jax.lax.stop_gradient(z_quantized - z_scaled)
        
        return z_ste

### `models/seer.py`

In [ ]:
%%writefile models/seer.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple

# Import the FSQ module we defined previously
from models.fsq import FSQ

class Seer(nn.Module):
    """
    The 'Hacker' or Prefrontal Cortex network.
    Observes the global state and generates a discrete compositional message.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 3
    lstm_features: int = 128
    num_message_heads: int = 1

    @nn.compact
    def __call__(
        self, 
        carry: Tuple[jnp.ndarray, jnp.ndarray], 
        map_obs: jnp.ndarray, 
        symbolic_obs: jnp.ndarray,
        target_images: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            map_obs: The Global Map View grid. Expected shape (batch, H, W, C).
            symbolic_obs: Guard Schedule + Sensor States. Expected shape (batch, features).
            target_images: The items the Doers must select. Expected shape (batch, 2, 5, 5, 3).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            discrete_message: The quantized $m_t$ vector sent to the Doer.
            thought_vector: The continuous pre-quantization vector.
            navigation_logits: Action logits used while the Seer is embodied.
        """
        
        # 1. Visual Encoder: CNN for the grid visual 
        x = nn.Conv(features=32, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(map_obs)
        x = nn.relu(x)
        x = nn.Conv(features=64, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(x)
        x = nn.relu(x)
        
        # Flatten visual features to a 1D vector per batch item
        # shape changes from (batch, H, W, channels) to (batch, H * W * channels)
        x_flat = x.reshape((x.shape[0], -1)) 
        
        # 2. Symbolic Encoder: MLP for symbolic data 
        y = nn.Dense(features=64)(symbolic_obs)
        y = nn.relu(y)
        
        # 3. Target Images Encoder
        # target_images shape is (batch, 2, 5, 5, 3)
        ti_flat = target_images.reshape((-1, 5, 5, 3))
        ti_conv1 = nn.Conv(features=16, kernel_size=(3, 3))(ti_flat)
        ti_conv1 = nn.relu(ti_conv1)
        ti_conv2 = nn.Conv(features=32, kernel_size=(3, 3))(ti_conv1)
        ti_conv2 = nn.relu(ti_conv2)
        ti_feats = ti_conv2.reshape((target_images.shape[0], -1))
        
        # 4. Fusion
        # Concatenate the visual, symbolic, and target features
        fused_features = jnp.concatenate([x_flat, y, ti_feats], axis=-1)
        
        # 5. Reasoning Module: LSTM 
        # Evaluates the current state in the context of the previous timestep [cite: 135]
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 5. Continuous Projection
        # Project LSTM hidden state to continuous vector z of size d.
        # Use Orthogonal init with higher scale to prevent FSQ mode collapse.
        thought_vector = nn.Dense(
            features=len(self.fsq_levels) * self.num_message_heads,
            kernel_init=nn.initializers.orthogonal(scale=2.0)
        )(lstm_out)
        thought_vector = thought_vector.reshape(
            (thought_vector.shape[0], self.num_message_heads, len(self.fsq_levels))
        )

        # 6. Output Head: FSQ Discretizer 
        # Transforms the continuous thought vector into the discrete message $m_t$ 
        fsq = FSQ(levels=self.fsq_levels)
        discrete_message = fsq(
            thought_vector.reshape((-1, thought_vector.shape[-1]))
        ).reshape(thought_vector.shape)

        if self.num_message_heads == 1:
            thought_vector = thought_vector[:, 0, :]
            discrete_message = discrete_message[:, 0, :]

        # During the pretraining phase the Seer physically navigates the grid.
        navigation_logits = nn.Dense(features=self.num_actions)(lstm_out)

        return new_carry, discrete_message, thought_vector, navigation_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )


### `models/doer.py`

In [ ]:
%%writefile models/doer.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple

class Doer(nn.Module):
    """
    The 'Thief' or Motor Cortex network.
    Operates on local observations and executes commands via discrete actions.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 9 # e.g., Move N/S/E/W, Toggle, Pick 0/1/2/3
    lstm_features: int = 128
    embed_dim: int = 16

    @nn.compact
    def __call__(
        self,
        carry: Tuple[jnp.ndarray, jnp.ndarray],
        local_obs: jnp.ndarray,
        proprioception: jnp.ndarray,
        message: jnp.ndarray,
        menu_images: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            local_obs: Egocentric 3x3 grid view. Expected shape (batch, 3, 3, C) or zeros.
            proprioception: Internal states (e.g., carrying item). Expected shape (batch, features).
            message: The quantized $m_t$ vector from the Seer. Expected shape (batch, d).
            menu_images: The 4 option images. Expected shape (batch, 4, 5, 5, 3).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            action_logits: Unnormalized log probabilities for the discrete action space.
        """
        
        # 1. Message Encoder: Learned Lookup Table
        # To preserve gradients, we MUST NOT cast to int and use nn.Embed directly 
        # Instead, FSQ naturally provides continuous coordinates (that happen to be quantized).
        # We can just linearly project this entire vector directly into a latent space!
        message_features = nn.Dense(features=self.embed_dim * len(self.fsq_levels))(message)
        message_features = nn.relu(message_features)
        
        # 2. Local Visual Encoder
        # Even for a small 3x3 grid, a single convolution or a dense layer extracts features
        x = nn.Conv(features=16, kernel_size=(2, 2))(local_obs)
        x = nn.relu(x)
        x_flat = x.reshape((x.shape[0], -1))
        
        # 3. Proprioception Encoder
        p = nn.Dense(features=16)(proprioception)
        p = nn.relu(p)
        
        # 4. Menu Images Encoder
        # menu_images shape is (batch, 4, 5, 5, 3)
        mi_flat = menu_images.reshape((-1, 5, 5, 3))
        mi_conv1 = nn.Conv(features=16, kernel_size=(3, 3))(mi_flat)
        mi_conv1 = nn.relu(mi_conv1)
        mi_conv2 = nn.Conv(features=32, kernel_size=(3, 3))(mi_conv1)
        mi_conv2 = nn.relu(mi_conv2)
        mi_feats = mi_conv2.reshape((menu_images.shape[0], -1))
        
        # 5. Fusion
        # Combine local vision, proprioception, menu images, and the embedded command
        fused_features = jnp.concatenate([x_flat, p, mi_feats, message_features], axis=-1)
        
        # 6. Reasoning Module: LSTM
        # Critical for integrating sequences of commands (e.g., maintaining a "Wait" state)
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 6. Output Head: Discrete Action Space
        # Projects the LSTM memory state into logits for physical actions
        action_logits = nn.Dense(features=self.num_actions)(lstm_out)
        
        return new_carry, action_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )

In [ ]:
import os; os.makedirs('envs', exist_ok=True); open('envs/__init__.py', 'w').close()

### `envs/two_doer_grid.py`

In [ ]:
%%writefile envs/two_doer_grid.py
import jax
import jax.numpy as jnp
import numpy as np
from flax import struct

UNSET_TWO_DOER_POSITIONS = jnp.full((2, 2), -1, dtype=jnp.int32)


@struct.dataclass
class TwoDoerState:
    positions: jnp.ndarray
    goals: jnp.ndarray
    step_count: jnp.ndarray
    done: jnp.ndarray
    target_items: jnp.ndarray
    shuffled_menus: jnp.ndarray
    selected_correctly: jnp.ndarray
    first_selection_correct: jnp.ndarray
    has_selected: jnp.ndarray
    has_arrived: jnp.ndarray
    selection_attempts: jnp.ndarray
    selected_option_idx: jnp.ndarray


class TwoDoerBottleneckEnv:
    """Two embodied Doers must swap sides through a single choke point."""

    def __init__(
        self,
        grid_height: int = 10,
        grid_width: int = 12,
        local_view_size: int = 3,
        corridor_length: int = 3,
        max_steps: int = 48,
        progress_reward_scale: float = 0.05,
        goal_reward: float = 1.0,
        arrival_reward: float = 0.5,
        individual_selection_reward: float = 0.5,
        wrong_selection_penalty: float = 0.15,
        wrong_selection_penalty_after_first: float | None = None,
        step_penalty: float = 0.03,
        wall_penalty: float = 0.02,
        collision_penalty: float = 0.05,
        doer_perception_level: int = 2,
        selection_phase_level: int = 1,
        render_tile_size: int = 24,
        max_selection_attempts: int = 4,
        pick_object_max_steps: int = 8,
        pick_object_listen_steps: int = 1,
    ):
        if grid_height < 8:
            raise ValueError("grid_height must be >= 8.")
        if grid_width < 10:
            raise ValueError("grid_width must be >= 10.")
        if local_view_size % 2 == 0:
            raise ValueError("local_view_size must be odd.")
        if corridor_length < 1:
            raise ValueError("corridor_length must be >= 1.")
        if grid_width - corridor_length - 2 < 4:
            raise ValueError("grid_width is too small for the requested corridor_length.")

        self.grid_height = int(grid_height)
        self.grid_width = int(grid_width)
        self.local_view_size = int(local_view_size)
        self.corridor_length = int(corridor_length)
        self.max_steps = int(max_steps)
        self.progress_reward_scale = jnp.asarray(progress_reward_scale, dtype=jnp.float32)
        self.goal_reward = jnp.asarray(goal_reward, dtype=jnp.float32)
        self.arrival_reward = jnp.asarray(arrival_reward, dtype=jnp.float32)
        self.individual_selection_reward = jnp.asarray(individual_selection_reward, dtype=jnp.float32)
        self.wrong_selection_penalty = jnp.asarray(wrong_selection_penalty, dtype=jnp.float32)
        if wrong_selection_penalty_after_first is None:
            wrong_selection_penalty_after_first = wrong_selection_penalty * 2.0
        self.wrong_selection_penalty_after_first = jnp.asarray(
            wrong_selection_penalty_after_first,
            dtype=jnp.float32,
        )
        self.step_penalty = jnp.asarray(step_penalty, dtype=jnp.float32)
        self.wall_penalty = jnp.asarray(wall_penalty, dtype=jnp.float32)
        self.collision_penalty = jnp.asarray(collision_penalty, dtype=jnp.float32)
        self.doer_perception_level = int(doer_perception_level)
        self._max_selection_attempts = int(max_selection_attempts)
        self.pick_object_max_steps = int(pick_object_max_steps)
        self.pick_object_listen_steps = int(pick_object_listen_steps)
        self.selection_phase_level = 1
        self.set_selection_phase_level(selection_phase_level)
        self.render_tile_size = int(render_tile_size)
        self.num_doers = 2
        self._view_radius = self.local_view_size // 2
        self._inner_width = self.grid_width - 2
        self._room_width = (self._inner_width - self.corridor_length) // 2
        self._extra_width = self._inner_width - self.corridor_length - 2 * self._room_width
        self._left_room_start_col = 1
        self._left_room_end_col = self._left_room_start_col + self._room_width
        self._corridor_row = self.grid_height // 2
        self._corridor_start_col = self._left_room_end_col
        self._corridor_end_col = self._corridor_start_col + self.corridor_length
        self._right_room_start_col = self._corridor_end_col
        self._right_room_end_col = self._right_room_start_col + self._room_width
        self._extra_wall_start_col = self._right_room_end_col
        self._extra_wall_end_col = self.grid_width - 1
        self._left_col = 1
        self._right_col = self._right_room_end_col - 1
        self._candidate_rows = jnp.asarray([1, 2, self.grid_height - 3, self.grid_height - 2], dtype=jnp.int32)
        self._wall_map = self._build_wall_map()
        self._goal_colors = jnp.asarray(
            [
                [0.90, 0.25, 0.25],
                [0.20, 0.70, 0.30],
            ],
            dtype=jnp.float32,
        )
        self._agent_colors = jnp.asarray(
            [
                [0.85, 0.10, 0.10],
                [0.05, 0.45, 0.95],
            ],
            dtype=jnp.float32,
        )
        self.item_bank = self._build_item_bank()

    @property
    def max_selection_attempts(self) -> int:
        return self._max_selection_attempts

    @property
    def phase_name(self) -> str:
        return "Pick_Object" if self.selection_phase_level == 1 else "Full_Training"

    @property
    def active_message_bits(self) -> int:
        return 4

    @property
    def is_pick_object_phase(self) -> bool:
        return self.selection_phase_level == 1

    @property
    def phase_max_steps(self) -> int:
        return self.pick_object_max_steps if self.is_pick_object_phase else self.max_steps

    def set_selection_phase_level(self, level: int) -> None:
        level = int(level)
        if level not in (1, 2):
            raise ValueError(f"Unsupported selection_phase_level={level}. Use 1 or 2.")
        self.selection_phase_level = level

    @property
    def num_actions(self) -> int:
        # stay, up, right, down, left, pick_0, pick_1, pick_2, pick_3
        return 9
        
    def _build_item_bank(self) -> jnp.ndarray:
        colors = jnp.asarray([
            [1.0, 0.2, 0.2],
            [0.2, 1.0, 0.2],
            [0.2, 0.4, 1.0],
            [1.0, 1.0, 0.2],
        ], dtype=jnp.float32)
        shape0 = jnp.ones((5, 5), dtype=jnp.float32)
        shape1 = jnp.zeros((5, 5), dtype=jnp.float32)
        shape1 = shape1.at[2, 1:4].set(1.0)
        shape1 = shape1.at[1:4, 2].set(1.0)
        shape2 = jnp.zeros((5, 5), dtype=jnp.float32)
        shape2 = shape2.at[1, 1].set(1.0)
        shape2 = shape2.at[2, 2].set(1.0)
        shape2 = shape2.at[3, 3].set(1.0)
        shape2 = shape2.at[1, 3].set(1.0)
        shape2 = shape2.at[3, 1].set(1.0)
        shape3 = jnp.ones((5, 5), dtype=jnp.float32)
        shape3 = shape3.at[1:4, 1:4].set(0.0)
        shapes = jnp.stack([shape0, shape1, shape2, shape3])
        bank = jnp.zeros((16, 5, 5, 3), dtype=jnp.float32)
        for c in range(4):
            for s in range(4):
                bank = bank.at[c * 4 + s].set(shapes[s, :, :, None] * colors[c])
        return bank

    def _build_wall_map(self) -> jnp.ndarray:
        wall_map = jnp.zeros((self.grid_height, self.grid_width), dtype=bool)
        wall_map = wall_map.at[0, :].set(True)
        wall_map = wall_map.at[-1, :].set(True)
        wall_map = wall_map.at[:, 0].set(True)
        wall_map = wall_map.at[:, -1].set(True)
        corridor_cols = jnp.arange(self._corridor_start_col, self._corridor_end_col)
        wall_map = wall_map.at[:, corridor_cols].set(True)
        wall_map = wall_map.at[self._corridor_row, corridor_cols].set(False)
        if self._extra_width > 0:
            extra_cols = jnp.arange(self._extra_wall_start_col, self._extra_wall_end_col)
            wall_map = wall_map.at[:, extra_cols].set(True)
        return wall_map

    @staticmethod
    def _manhattan_distance(positions: jnp.ndarray, goals: jnp.ndarray) -> jnp.ndarray:
        return jnp.abs(positions - goals).sum(axis=-1).astype(jnp.float32)

    def _compose_global_map(self, state: TwoDoerState) -> jnp.ndarray:
        global_map = jnp.zeros((self.grid_height, self.grid_width, 5), dtype=jnp.float32)
        global_map = global_map.at[:, :, 0].set(self._wall_map.astype(jnp.float32))
        global_map = global_map.at[state.positions[0, 0], state.positions[0, 1], 1].set(1.0)
        global_map = global_map.at[state.positions[1, 0], state.positions[1, 1], 2].set(1.0)
        global_map = global_map.at[state.goals[0, 0], state.goals[0, 1], 3].set(1.0)
        global_map = global_map.at[state.goals[1, 0], state.goals[1, 1], 4].set(1.0)
        return global_map

    def _extract_local_views(self, global_map: jnp.ndarray, positions: jnp.ndarray) -> jnp.ndarray:
        padded_map = jnp.pad(
            global_map,
            (
                (self._view_radius, self._view_radius),
                (self._view_radius, self._view_radius),
                (0, 0),
            ),
        )

        def slice_agent_view(position):
            row = position[0]
            col = position[1]
            return jax.lax.dynamic_slice(
                padded_map,
                (row, col, 0),
                (self.local_view_size, self.local_view_size, global_map.shape[-1]),
            )

        return jax.vmap(slice_agent_view)(positions)

    def _split_observations(self, state: TwoDoerState):
        global_map = self._compose_global_map(state)
        full_local_views = self._extract_local_views(global_map, state.positions)
        goals_reached = jnp.all(state.positions == state.goals, axis=-1).astype(jnp.float32)
        agent_identity = jnp.eye(self.num_doers, dtype=jnp.float32)
        position_features = jnp.stack(
            [
                state.positions[:, 0].astype(jnp.float32) / float(self.grid_height - 1),
                state.positions[:, 1].astype(jnp.float32) / float(self.grid_width - 1),
            ],
            axis=-1,
        )
        proprioception_full = jnp.concatenate(
            [position_features, agent_identity, goals_reached[:, None]],
            axis=-1,
        )
        if self.doer_perception_level == 2:
            local_views = jnp.zeros_like(full_local_views)
            proprioceptions = proprioception_full
        elif self.doer_perception_level == 3:
            local_views = jnp.zeros_like(full_local_views)
            proprioceptions = jnp.concatenate(
                [
                    jnp.zeros_like(position_features),
                    agent_identity,
                    goals_reached[:, None],
                ],
                axis=-1,
            )
        else:
            raise ValueError(
                f"Unsupported doer_perception_level={self.doer_perception_level}. "
                "Use 2 or 3."
            )
        symbolic_state = jnp.concatenate(
            [
                jnp.asarray(
                    [
                        state.positions[0, 0] / float(self.grid_height - 1),
                        state.positions[0, 1] / float(self.grid_width - 1),
                        state.positions[1, 0] / float(self.grid_height - 1),
                        state.positions[1, 1] / float(self.grid_width - 1),
                    ],
                    dtype=jnp.float32,
                ),
                jnp.asarray(
                    [
                        state.goals[0, 0] / float(self.grid_height - 1),
                        state.goals[0, 1] / float(self.grid_width - 1),
                        state.goals[1, 0] / float(self.grid_height - 1),
                        state.goals[1, 1] / float(self.grid_width - 1),
                    ],
                    dtype=jnp.float32,
                ),
                goals_reached,
                jnp.asarray(
                    [state.step_count.astype(jnp.float32) / float(self.max_steps)],
                    dtype=jnp.float32,
                ),
            ],
            axis=0,
        )
        
        target_images = self.item_bank[state.target_items]
        menu_images = self.item_bank[state.shuffled_menus]
        menu_images = jnp.where(goals_reached[:, None, None, None, None], menu_images, 0.0)
        pick_available = jnp.logical_and(
            goals_reached.astype(bool),
            jnp.logical_or(
                jnp.asarray(not self.is_pick_object_phase),
                state.step_count >= self.pick_object_listen_steps,
            ),
        )

        return {
            "global_map": global_map,
            "symbolic_state": symbolic_state,
            "local_views": local_views,
            "proprioceptions": proprioceptions,
            "target_images": target_images,
            "menu_images": menu_images,
            "pick_available": pick_available,
        }

    def _goals_from_positions(self, positions: jnp.ndarray) -> jnp.ndarray:
        return jnp.asarray(
            [
                [positions[0, 0], self._right_col],
                [positions[1, 0], self._left_col],
            ],
            dtype=jnp.int32,
        )

    def reset(
        self,
        key: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        row_key_a, row_key_b = jax.random.split(key)
        row_a = self._candidate_rows[jax.random.randint(row_key_a, (), 0, self._candidate_rows.shape[0])]
        row_b = self._candidate_rows[jax.random.randint(row_key_b, (), 0, self._candidate_rows.shape[0])]
        sampled_positions = jnp.asarray(
            [
                [row_a, self._left_col],
                [row_b, self._right_col],
            ],
            dtype=jnp.int32,
        )
        start_positions = jnp.where(
            jnp.all(fixed_positions >= 0),
            fixed_positions.astype(jnp.int32),
            sampled_positions,
        )
        phase_goals = self._goals_from_positions(start_positions)
        positions = jnp.where(
            self.is_pick_object_phase,
            phase_goals,
            start_positions,
        )
        goals = jnp.where(
            self.is_pick_object_phase,
            positions,
            phase_goals,
        )
        
        key_target_a, key_target_b, key_menu_a, key_menu_b = jax.random.split(key, 4)
        target_a = jax.random.randint(key_target_a, (), 0, 16)
        target_b = jax.random.randint(key_target_b, (), 0, 16)
        
        def make_menu(rng_key, target):
            p = jnp.where(jnp.arange(16) == target, 0.0, 1.0 / 15.0)
            distractors = jax.random.choice(rng_key, jnp.arange(16), shape=(3,), replace=False, p=p)
            menu = jnp.concatenate([jnp.array([target]), distractors])
            _, shuffle_key = jax.random.split(rng_key)
            return jax.random.permutation(shuffle_key, menu)
            
        menu_a = make_menu(key_menu_a, target_a)
        menu_b = make_menu(key_menu_b, target_b)
        
        state = TwoDoerState(
            positions=positions,
            goals=goals,
            step_count=jnp.asarray(0, dtype=jnp.int32),
            done=jnp.asarray(False),
            target_items=jnp.array([target_a, target_b]),
            shuffled_menus=jnp.stack([menu_a, menu_b]),
            selected_correctly=jnp.array([False, False]),
            first_selection_correct=jnp.array([False, False]),
            has_selected=jnp.array([False, False]),
            has_arrived=jnp.array([False, False]),
            selection_attempts=jnp.zeros((self.num_doers,), dtype=jnp.int32),
            selected_option_idx=jnp.array([-1, -1]),
        )
        return self._split_observations(state), state

    def reset_batch(
        self,
        keys: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        return jax.vmap(self.reset, in_axes=(0, None))(keys, fixed_positions)

    def _resolve_actions(self, positions: jnp.ndarray, actions: jnp.ndarray):
        deltas = jnp.asarray(
            [
                [0, 0],
                [-1, 0],
                [0, 1],
                [1, 0],
                [0, -1],
            ],
            dtype=jnp.int32,
        )
        raw_targets = positions + deltas[actions]
        target_walls = self._wall_map[raw_targets[:, 0], raw_targets[:, 1]]
        wall_hits = jnp.logical_and(actions != 0, target_walls)
        proposed = jnp.where(target_walls[:, None], positions, raw_targets)

        same_target = jnp.all(proposed[0] == proposed[1])
        swap_positions = jnp.logical_and(
            jnp.all(proposed[0] == positions[1]),
            jnp.all(proposed[1] == positions[0]),
        )
        a_into_b = jnp.logical_and(
            jnp.all(proposed[0] == positions[1]),
            jnp.all(proposed[1] == positions[1]),
        )
        b_into_a = jnp.logical_and(
            jnp.all(proposed[1] == positions[0]),
            jnp.all(proposed[0] == positions[0]),
        )
        collision = jnp.logical_or(jnp.logical_or(same_target, swap_positions), jnp.logical_or(a_into_b, b_into_a))

        collision_blocks = jnp.asarray(
            [
                jnp.logical_or(same_target, jnp.logical_or(swap_positions, a_into_b)),
                jnp.logical_or(same_target, jnp.logical_or(swap_positions, b_into_a)),
            ],
            dtype=bool,
        )
        final_positions = jnp.where(collision_blocks[:, None], positions, proposed)
        return final_positions, wall_hits, collision_blocks

    def step(
        self,
        key: jnp.ndarray,
        state: TwoDoerState,
        actions: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        def reset_branch(_):
            reset_obs, reset_state = self.reset(key, fixed_positions=fixed_positions)
            zeros = jnp.asarray(0.0, dtype=jnp.float32)
            info = {
                "task_reward": zeros,
                "individual_selection_reward": zeros,
                "wrong_selection_penalty": zeros,
                "valid_selection_count": zeros,
                "correct_selection_count": zeros,
                "progress_reward_per_doer": jnp.zeros((self.num_doers,), dtype=jnp.float32),
                "step_penalty": zeros,
                "wall_penalty": zeros,
                "collision_penalty": zeros,
                "goal_distance": self._manhattan_distance(reset_state.positions, reset_state.goals),
                "success": jnp.asarray(False),
                "eventual_success": jnp.asarray(False),
                "first_try_success": jnp.asarray(False),
                "failed": jnp.asarray(False),
            }
            return reset_obs, reset_state, zeros, jnp.asarray(False), info

        def step_branch(_):
            phase_max_steps = self.phase_max_steps
            nav_actions = jnp.where(actions < 5, actions, 0)
            select_actions = actions - 5
            is_selecting = actions >= 5
            at_goal = jnp.all(state.positions == state.goals, axis=-1)
            pick_available = jnp.logical_or(
                ~self.is_pick_object_phase,
                state.step_count >= self.pick_object_listen_steps,
            )

            valid_selection = jnp.logical_and(is_selecting, at_goal)
            valid_selection = jnp.logical_and(valid_selection, pick_available)
            valid_selection = jnp.logical_and(valid_selection, ~state.has_selected)

            chosen_item = jnp.take_along_axis(state.shuffled_menus, select_actions[:, None], axis=1)[:, 0]
            is_correct = chosen_item == state.target_items
            
            new_selection_attempts = state.selection_attempts + valid_selection.astype(jnp.int32)
            first_selection = jnp.logical_and(valid_selection, state.selection_attempts == 0)
            correct_selection = jnp.logical_and(first_selection, is_correct)
            wrong_selection = jnp.logical_and(first_selection, ~is_correct)
            new_selected_correctly = jnp.logical_or(state.selected_correctly, correct_selection)
            new_first_selection_correct = jnp.logical_or(
                state.first_selection_correct,
                correct_selection,
            )
            new_has_selected = jnp.logical_or(
                state.has_selected,
                first_selection,
            )

            if self.is_pick_object_phase:
                next_positions = state.positions
                new_distances = self._manhattan_distance(next_positions, state.goals)
                progress_reward_per_doer = jnp.zeros((self.num_doers,), dtype=jnp.float32)
                progress_reward = jnp.asarray(0.0, dtype=jnp.float32)
                wall_penalty = jnp.asarray(0.0, dtype=jnp.float32)
                collision_penalty = jnp.asarray(0.0, dtype=jnp.float32)
                new_has_arrived = jnp.ones((self.num_doers,), dtype=bool)
                arrival_reward = jnp.asarray(0.0, dtype=jnp.float32)
            else:
                old_distances = self._manhattan_distance(state.positions, state.goals)
                next_positions, wall_hits, collision_blocks = self._resolve_actions(state.positions, nav_actions)

                # Lock position if selecting on this step or if already selected previously
                is_locked = jnp.logical_or(is_selecting, state.has_selected)
                next_positions = jnp.where(is_locked[:, None], state.positions, next_positions)

                new_distances = self._manhattan_distance(next_positions, state.goals)
                progress_reward_per_doer = (old_distances - new_distances) * self.progress_reward_scale
                progress_reward = progress_reward_per_doer.sum()
                wall_penalty = self.wall_penalty * wall_hits.astype(jnp.float32).sum()
                collision_penalty = (
                    self.collision_penalty * collision_blocks.astype(jnp.float32).sum()
                )

                arrived_this_step = jnp.all(next_positions == state.goals, axis=-1)
                new_has_arrived = jnp.logical_or(state.has_arrived, arrived_this_step)

                team_just_arrived = jnp.logical_and(
                    jnp.all(new_has_arrived),
                    ~jnp.all(state.has_arrived)
                )
                arrival_reward = jnp.where(
                    team_just_arrived,
                    self.arrival_reward,
                    jnp.asarray(0.0, dtype=jnp.float32),
                )
            
            new_selected_option_idx = jnp.where(
                valid_selection,
                select_actions,
                state.selected_option_idx
            )

            individual_selection_reward = (
                correct_selection.astype(jnp.float32) * self.individual_selection_reward
            ).sum()
            wrong_selection_penalty = (
                wrong_selection.astype(jnp.float32) * self.wrong_selection_penalty
            ).sum()
            valid_selection_count = valid_selection.astype(jnp.float32).sum()
            correct_selection_count = correct_selection.astype(jnp.float32).sum()

            all_terminal = jnp.all(new_has_selected)
            all_success = jnp.logical_and(all_terminal, jnp.all(new_first_selection_correct))
            first_try_success = all_success
            
            team_completion_reward = jnp.where(
                all_success,
                self.goal_reward,
                jnp.asarray(0.0, dtype=jnp.float32),
            )
            task_reward = team_completion_reward + individual_selection_reward

            reward = (
                task_reward
                + progress_reward
                + arrival_reward
                - jnp.where(
                    jnp.logical_and(self.is_pick_object_phase, state.step_count < self.pick_object_listen_steps),
                    jnp.asarray(0.0, dtype=jnp.float32),
                    self.step_penalty,
                )
                - wrong_selection_penalty
                - wall_penalty
                - collision_penalty
            )
            next_state = TwoDoerState(
                positions=next_positions,
                goals=state.goals,
                step_count=state.step_count + 1,
                done=jnp.logical_or(all_terminal, state.step_count + 1 >= phase_max_steps),
                target_items=state.target_items,
                shuffled_menus=state.shuffled_menus,
                selected_correctly=new_selected_correctly,
                first_selection_correct=new_first_selection_correct,
                has_selected=new_has_selected,
                has_arrived=new_has_arrived,
                selection_attempts=new_selection_attempts,
                selected_option_idx=new_selected_option_idx,
            )
            info = {
                "task_reward": team_completion_reward,
                "individual_selection_reward": individual_selection_reward,
                "wrong_selection_penalty": wrong_selection_penalty,
                "valid_selection_count": valid_selection_count,
                "correct_selection_count": correct_selection_count,
                "progress_reward_per_doer": progress_reward_per_doer,
                "step_penalty": jnp.where(
                    jnp.logical_and(self.is_pick_object_phase, state.step_count < self.pick_object_listen_steps),
                    jnp.asarray(0.0, dtype=jnp.float32),
                    self.step_penalty,
                ),
                "wall_penalty": wall_penalty,
                "collision_penalty": collision_penalty,
                "goal_distance": new_distances,
                "success": all_success,
                "eventual_success": all_success,
                "first_try_success": first_try_success,
                "failed": jnp.logical_and(next_state.done, ~all_success),
            }
            return self._split_observations(next_state), next_state, reward, next_state.done, info

        return jax.lax.cond(state.done, reset_branch, step_branch, operand=None)

    def step_batch(
        self,
        keys: jnp.ndarray,
        states: TwoDoerState,
        actions: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        return jax.vmap(self.step, in_axes=(0, 0, 0, None))(keys, states, actions, fixed_positions)

    def render(self, state: TwoDoerState) -> np.ndarray:
        tile = self.render_tile_size
        frame = np.ones((self.grid_height * tile, self.grid_width * tile, 3), dtype=np.float32) * 0.96

        wall_color = np.array([0.18, 0.18, 0.22], dtype=np.float32)
        corridor_color = np.array([0.98, 0.88, 0.45], dtype=np.float32)

        wall_map = np.asarray(self._wall_map)
        for row in range(self.grid_height):
            for col in range(self.grid_width):
                y0 = row * tile
                y1 = (row + 1) * tile
                x0 = col * tile
                x1 = (col + 1) * tile
                if wall_map[row, col]:
                    frame[y0:y1, x0:x1] = wall_color

        for col in range(self._corridor_start_col, self._corridor_end_col):
            y0 = self._corridor_row * tile
            y1 = (self._corridor_row + 1) * tile
            x0 = col * tile
            x1 = (col + 1) * tile
            frame[y0:y1, x0:x1] = corridor_color

        for agent_idx in range(self.num_doers):
            goal_row, goal_col = np.asarray(state.goals[agent_idx]).tolist()
            y0 = goal_row * tile
            x0 = goal_col * tile
            inset = 2
            frame[
                y0 + inset:(goal_row + 1) * tile - inset,
                x0 + inset:(goal_col + 1) * tile - inset,
            ] = np.asarray(self._agent_colors[agent_idx])

        yy, xx = np.mgrid[0:tile, 0:tile]
        left_triangle = np.logical_and(xx >= tile // 5, np.abs(yy - tile // 2) <= xx - tile // 5)
        right_triangle = np.logical_and(
            xx <= tile - tile // 5 - 1,
            np.abs(yy - tile // 2) <= (tile - tile // 5 - 1) - xx,
        )

        for agent_idx in range(self.num_doers):
            row, col = np.asarray(state.positions[agent_idx]).tolist()
            y0 = row * tile
            x0 = col * tile
            tile_view = frame[y0:(row + 1) * tile, x0:(col + 1) * tile]
            triangle_mask = right_triangle if agent_idx == 0 else left_triangle
            tile_view[triangle_mask] = np.asarray(self._agent_colors[agent_idx])
            frame[y0:(row + 1) * tile, x0:(col + 1) * tile] = tile_view

        target_ring = np.array([0.95, 0.75, 0.10], dtype=np.float32)
        correct_ring = np.array([0.18, 0.72, 0.28], dtype=np.float32)
        wrong_ring = np.array([0.88, 0.22, 0.22], dtype=np.float32)

        # Render target and menu options on the outer columns.
        bank = np.asarray(self.item_bank)
        for agent_idx in range(self.num_doers):
            offset_y, offset_x = (tile - 15) // 2, (tile - 15) // 2

            side_col = (self.grid_width - 1) * tile if agent_idx == 0 else 0
            target_id = int(np.asarray(state.target_items[agent_idx]))
            target_img = bank[target_id]
            target_upscaled = np.repeat(np.repeat(target_img, 3, axis=0), 3, axis=1)
            target_y0 = tile
            frame[target_y0 + offset_y - 2: target_y0 + offset_y + 17, side_col + offset_x - 2: side_col + offset_x + 17] = target_ring
            frame[target_y0 + offset_y: target_y0 + offset_y + 15, side_col + offset_x: side_col + offset_x + 15] = target_upscaled

            menus = np.asarray(state.shuffled_menus[agent_idx])
            has_sel = bool(np.asarray(state.selection_attempts[agent_idx] > 0))
            sel_idx = int(np.asarray(state.selected_option_idx[agent_idx]))
            
            for m_idx, option_id in enumerate(menus):
                opt_img = bank[option_id]
                opt_upscaled = np.repeat(np.repeat(opt_img, 3, axis=0), 3, axis=1)

                my0 = (3 + m_idx) * tile
                mx0 = side_col

                if int(option_id) == target_id:
                    frame[
                        my0 + offset_y - 1: my0 + offset_y + 16,
                        mx0 + offset_x - 1: mx0 + offset_x + 16,
                    ] = target_ring
                if has_sel and sel_idx == m_idx:
                    picked_ring = correct_ring if int(option_id) == target_id else wrong_ring
                    frame[
                        my0 + offset_y - 2: my0 + offset_y + 17,
                        mx0 + offset_x - 2: mx0 + offset_x + 17,
                    ] = picked_ring

                frame[my0 + offset_y: my0 + offset_y + 15, mx0 + offset_x: mx0 + offset_x + 15] = opt_upscaled

        return (frame * 255.0).astype(np.uint8)


### `training/action_masking.py`

In [ ]:
%%writefile training/action_masking.py
import jax.numpy as jnp


def mask_pick_actions_until_menu_visible(
    logits: jnp.ndarray,
    menu_images: jnp.ndarray,
    pick_only_phase: bool = False,
    pick_available: jnp.ndarray | None = None,
) -> jnp.ndarray:
    """Disable invalid picks and optionally freeze navigation during the pick-only phase."""
    menu_feature_axes = tuple(range(logits.ndim - 1, menu_images.ndim))
    menu_visible = jnp.any(menu_images > 0.0, axis=menu_feature_axes)
    if pick_available is not None:
        menu_visible = jnp.logical_and(menu_visible, pick_available)
    action_ids = jnp.arange(logits.shape[-1])
    pick_mask = action_ids >= 5
    invalid_pick_mask = jnp.logical_and(~menu_visible[..., None], pick_mask)
    large_negative = jnp.asarray(-1.0e9, dtype=logits.dtype)
    masked_logits = jnp.where(invalid_pick_mask, large_negative, logits)
    if pick_only_phase:
        navigation_mask = jnp.logical_and(action_ids >= 1, action_ids < 5)
        masked_logits = jnp.where(navigation_mask, large_negative, masked_logits)
    return masked_logits


### `training/message_masking.py`

In [ ]:
%%writefile training/message_masking.py
import jax.numpy as jnp


def hard_mask_inactive_message_bits(messages: jnp.ndarray, active_bits: int) -> jnp.ndarray:
    """Force trailing message dimensions to zero while keeping the architecture fixed."""
    bit_mask = (jnp.arange(messages.shape[-1]) < active_bits).astype(messages.dtype)
    return messages * bit_mask


### `training/gae.py`

In [ ]:
%%writefile training/gae.py
import jax
import jax.numpy as jnp
from typing import Tuple

def compute_gae(
    rewards: jnp.ndarray,
    values: jnp.ndarray,
    dones: jnp.ndarray,
    last_val: jnp.ndarray,
    gamma: float = 0.99,
    gae_lambda: float = 0.95
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    """
    Computes Generalized Advantage Estimation (GAE) and target returns.
    
    Args:
        rewards: Array of rewards collected during the rollout. Shape: (num_steps,)
        values: Array of value predictions from the Critic. Shape: (num_steps,)
        dones: Array of boolean/integer done flags. Shape: (num_steps,)
        last_val: The Critic's value prediction for the state *after* the final step.
        gamma: Discount factor.
        gae_lambda: Bias-variance tradeoff parameter for GAE.
        
    Returns:
        advantages: The calculated GAE advantages. Shape: (num_steps,)
        returns: The target values for the Critic to learn from (Advantages + Values).
    """
    
    def _gae_step(carry, transition_data):
        """A single step of the reverse scan."""
        gae_t_plus_1 = carry
        reward, value, done, next_value = transition_data
        
        # Calculate the Temporal Difference (TD) error
        # If the episode ended (done=1), the next state has no value.
        delta = reward + gamma * next_value * (1.0 - done) - value
        
        # Calculate the advantage for the current timestep
        gae_t = delta + gamma * gae_lambda * (1.0 - done) * gae_t_plus_1
        
        # Pass the current advantage back as the carry for the previous timestep
        return gae_t, gae_t

    # To calculate the TD error, we need the value of the next state for every step.
    # We create an array of "next values" by shifting the values array by one 
    # and appending the bootstrap 'last_val' at the end.
    next_values = jnp.append(values[1:], last_val)
    
    # Pack the data for the scan
    scan_data = (rewards, values, dones, next_values)
    
    # Initialize the carry with 0.0 (the advantage after the final step)
    initial_gae = 0.0
    
    # Run the scan in reverse to propagate advantages backwards
    _, advantages = jax.lax.scan(_gae_step, initial_gae, scan_data, reverse=True)
    
    # The return value (target for the critic) is simply the advantage + the predicted value
    returns = advantages + values
    
    return advantages, returns

### `training/loop.py`

In [ ]:
%%writefile training/loop.py
import jax
import jax.numpy as jnp
import distrax
from typing import Tuple, Any, Dict
from training.gae import compute_gae 
from training.action_masking import mask_pick_actions_until_menu_visible
from training.message_masking import hard_mask_inactive_message_bits

# Assuming Transition is imported from your mappo.py or a shared datatypes file
from agents.mappo import Transition, TwoDoerTransition
from eval.metrics import compute_cic

def make_rollout_step(
    env,
    seer_apply_fn,
    doer_apply_fn,
    critic_apply_fn,
    follow_reward_scale=0.1,
):
    """
    A closure that returns the JAX-compilable step function.
    Passing the environment and apply functions here avoids passing them 
    repeatedly into the compiled loop.
    """
    follow_reward_scale = jnp.asarray(follow_reward_scale, dtype=jnp.float32)
    
    def rollout_step(runner_state: Tuple, _):
        """
        Executes a single environment step and network forward pass.
        Designed to be passed directly to jax.lax.scan.
        """
        # Unpack the runner state
        (
            params,
            seer_carry,
            doer_carry,
            env_state,
            env_obs,
            rng,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        ) = runner_state
        num_envs = env_obs["global_map"].shape[0]
        communication_mode = control_mode == 1
        
        # Split the PRNG key for the stochastic actions
        rng, seer_rng, doer_rng, env_rng = jax.random.split(rng, 4)
        env_step_keys = jax.random.split(env_rng, num_envs)
        
        # 1. Seer Forward Pass (Prefrontal Cortex)
        # Enforcing CTDE: Seer gets the global view[cite: 131, 132].
        # In a custom jaxmarl wrapper, you would extract these from env_obs
        global_map = env_obs["global_map"]
        symbolic_obs = env_obs["symbolic_state"]
        
        next_seer_carry, discrete_message, _, seer_nav_logits = seer_apply_fn(
            {"params": params["seer"]}, 
            seer_carry, 
            global_map, 
            symbolic_obs,
            env_obs["target_images"]
        )
        
        # 2. Doer Forward Pass (Motor Cortex)
        # Enforcing Functional Asymmetry: Doer gets local view and the message[cite: 137, 138].
        local_obs = env_obs["local_view"]
        proprioception = env_obs["proprioception"]
        
        next_doer_carry, action_logits = doer_apply_fn(
            {"params": params["doer"]}, 
            doer_carry, 
            local_obs, 
            proprioception, 
            discrete_message,
            env_obs["menu_images"]
        )
        _, null_action_logits = doer_apply_fn(
            {"params": params["doer"]},
            doer_carry,
            local_obs,
            proprioception,
            jnp.zeros_like(discrete_message),
            env_obs["menu_images"]
        )
        
        # 3. Action Selection
        seer_pi = distrax.Categorical(logits=seer_nav_logits)
        pi = distrax.Categorical(logits=action_logits)
        seer_action = seer_pi.sample(seed=seer_rng)
        doer_action = pi.sample(seed=doer_rng)
        seer_log_prob = seer_pi.log_prob(seer_action)
        doer_log_prob = pi.log_prob(doer_action)
        env_action = jnp.where(communication_mode, doer_action, seer_action)
        message_action = jnp.argmax(action_logits, axis=-1)
        null_message_action = jnp.argmax(null_action_logits, axis=-1)
        action_change_bonus = (
            message_action != null_message_action
        ).astype(jnp.float32) * follow_reward_scale
        
        # 4. Critic Forward Pass (Centralized Training)
        # The critic evaluates the global state to guide learning[cite: 111].
        value = critic_apply_fn({"params": params["critic"]}, global_map)
        
        # 5. Environment Step
        # Step the jaxmarl environment using the chosen action
        # Note: jaxmarl expects a dictionary of actions for multi-agent, 
        # adapt this based on your specific wrapper implementation.
        next_env_obs, next_env_state, reward, done, info = env.step_batch(
            env_step_keys,
            env_state,
            env_action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )

        task_reward = info["task_reward"]
        progress_reward = info["progress_reward"]
        step_penalty = info["step_penalty"]
        bump_penalty = info["bump_penalty"]
        useful_communication = jnp.logical_or(
            progress_reward > 0.0,
            task_reward > 0.0,
        )
        follow_reward = jnp.where(
            jnp.logical_and(communication_mode, useful_communication),
            action_change_bonus,
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        shared_comm_reward = (
            task_reward + progress_reward + follow_reward - step_penalty - bump_penalty
        )
        seer_reward = jnp.where(
            communication_mode,
            shared_comm_reward,
            task_reward + progress_reward - step_penalty - bump_penalty,
        )
        doer_reward = jnp.where(
            communication_mode,
            shared_comm_reward,
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        reward = jnp.stack([seer_reward, doer_reward], axis=-1)

        done_mask = done[:, None]
        next_seer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_seer_carry,
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_doer_carry,
        )
        
        # 6. Build the Transition
        transition = Transition(
            global_obs=global_map,
            symbolic_obs=symbolic_obs,
            local_obs=local_obs,
            proprioception=proprioception,
            message=discrete_message,
            target_images=env_obs["target_images"],
            menu_images=env_obs["menu_images"],
            doer_action=doer_action,
            doer_log_prob=doer_log_prob,
            seer_action=seer_action,
            seer_log_prob=seer_log_prob,
            value=value,
            reward=reward,
            task_reward=task_reward,
            progress_reward=progress_reward,
            follow_reward=follow_reward,
            cic_reward_component=jnp.zeros_like(task_reward),
            cic_score=jnp.zeros_like(task_reward),
            step_penalty_component=step_penalty,
            bump_penalty_component=bump_penalty,
            done=done,
            # Advantage and return will be calculated post-rollout using GAE
            advantage=jnp.zeros_like(reward), 
            return_val=jnp.zeros_like(reward)
        )
        
        # Repack the updated runner state
        next_runner_state = (
            params,
            next_seer_carry,
            next_doer_carry,
            next_env_state,
            next_env_obs,
            rng,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )
        
        return next_runner_state, transition

    return rollout_step

import functools

@functools.partial(jax.jit, static_argnames=("num_steps", "step_fn", "critic_apply_fn", "doer_apply_fn"))
def generate_trajectory_and_gae(
    params, rng, env_obs, env_state, seer_carry, doer_carry, vision_radius: jnp.ndarray, control_mode: jnp.ndarray, fixed_goal_position: jnp.ndarray, fixed_start_position: jnp.ndarray, cic_coef: jnp.ndarray, num_steps: int,
    step_fn, critic_apply_fn, doer_apply_fn
):
    """
    Executes the full episode rollout and computes GAE in a single compiled pass.
    Note: We pass the pre-compiled step_fn and initial states directly here 
    for better JAX compilation efficiency.
    """
    initial_runner_state = (
        params,
        seer_carry,
        doer_carry,
        env_state,
        env_obs,
        rng,
        vision_radius,
        control_mode,
        fixed_goal_position,
        fixed_start_position,
    )
    
    # 1. Execute the scan loop to collect the raw trajectory
    final_runner_state, trajectory_batch = jax.lax.scan(
        step_fn, initial_runner_state, None, length=num_steps
    )
    
    # 2. Extract the final state for Critic bootstrapping
    # Unpack the final runner state to get the last env_obs
    _, _, _, _, final_env_obs, final_rng, _, _, _, _ = final_runner_state
    
    # Enforce CTDE: The critic evaluates the global map 
    final_global_map = final_env_obs["global_map"]
    
    # 3. Calculate the bootstrap value (last_val)
    # The critic evaluates the state *after* the final step
    last_val = critic_apply_fn({"params": params["critic"]}, final_global_map)
    
    communication_mode = control_mode == 1

    def add_cic_bonus(_):
        cic_score = compute_cic(
            doer_apply_fn,
            params["doer"],
            trajectory_batch,
            doer_carry,
            final_rng,
        )
        per_step_cic_reward = cic_coef * cic_score / jnp.asarray(num_steps, dtype=jnp.float32)
        cic_reward_component = jnp.full_like(trajectory_batch.task_reward, per_step_cic_reward)
        cic_score_component = jnp.full_like(trajectory_batch.task_reward, cic_score)
        reward_with_cic = trajectory_batch.reward + cic_reward_component[..., None]
        return reward_with_cic, cic_reward_component, cic_score_component

    def skip_cic_bonus(_):
        zeros = jnp.zeros_like(trajectory_batch.task_reward)
        return trajectory_batch.reward, zeros, zeros

    reward_with_cic, cic_reward_component, cic_score_component = jax.lax.cond(
        jnp.logical_and(communication_mode, cic_coef > 0.0),
        add_cic_bonus,
        skip_cic_bonus,
        operand=None,
    )
    
    # 5. Compute GAE
    # Note: If you scale up to multiple environments (num_envs > 1), you would wrap 
    # compute_gae with jax.vmap(compute_gae, in_axes=1, out_axes=1) to vectorize 
    # the advantage calculation across all environments simultaneously.
    advantages, returns = jax.vmap(
        jax.vmap(
            compute_gae,
            in_axes=(1, 1, 1, 0, None, None),
            out_axes=1,
        ),
        in_axes=(2, 2, None, 1, None, None),
        out_axes=2,
    )(
        reward_with_cic,
        trajectory_batch.value,
        trajectory_batch.done,
        last_val,
        0.99,
        0.95,
    )
    
    # 5. Update the trajectory batch
    # Using the .replace() method provided by chex/flax dataclasses
    trajectory_batch = trajectory_batch.replace(
        reward=reward_with_cic,
        cic_reward_component=cic_reward_component,
        cic_score=cic_score_component,
        advantage=advantages,
        return_val=returns
    )
    
    return final_runner_state, trajectory_batch


def make_two_doer_rollout_step(
    env,
    seer_apply_fn,
    doer_apply_fn,
    critic_apply_fn,
):
    """Rollout step for one Seer coordinating two embodied Doers."""

    def rollout_step(runner_state: Tuple, _):
        params, seer_carry, doer_carry, env_state, env_obs, rng, fixed_positions = runner_state
        num_envs = env_obs["global_map"].shape[0]
        global_map = env_obs["global_map"]
        symbolic_obs = env_obs["symbolic_state"]
        local_obs = env_obs["local_views"]
        proprioception = env_obs["proprioceptions"]

        rng, action_rng, env_rng = jax.random.split(rng, 3)
        env_step_keys = jax.random.split(env_rng, num_envs)

        next_seer_carry, discrete_messages, _, _ = seer_apply_fn(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_obs,
            env_obs["target_images"]
        )
        discrete_messages = hard_mask_inactive_message_bits(
            discrete_messages,
            active_bits=env.active_message_bits,
        )

        batch_size, num_doers = local_obs.shape[:2]
        flat_local_obs = local_obs.reshape((batch_size * num_doers,) + local_obs.shape[2:])
        flat_proprioception = proprioception.reshape(
            (batch_size * num_doers,) + proprioception.shape[2:]
        )
        flat_messages = discrete_messages.reshape(
            (batch_size * num_doers,) + discrete_messages.shape[2:]
        )
        flat_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_doer_carry, flat_logits = doer_apply_fn(
            {"params": params["doer"]},
            flat_doer_carry,
            flat_local_obs,
            flat_proprioception,
            flat_messages,
            env_obs["menu_images"].reshape((batch_size * num_doers,) + env_obs["menu_images"].shape[2:])
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_doer_carry,
        )
        doer_logits = flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))
        doer_logits = mask_pick_actions_until_menu_visible(
            doer_logits,
            env_obs["menu_images"],
            pick_only_phase=env.is_pick_object_phase,
            pick_available=env_obs["pick_available"],
        )
        doer_pi = distrax.Categorical(logits=doer_logits)
        doer_action = doer_pi.sample(seed=action_rng)
        doer_log_prob = doer_pi.log_prob(doer_action)
        value = critic_apply_fn({"params": params["critic"]}, global_map).squeeze(-1)

        next_env_obs, next_env_state, reward, done, info = env.step_batch(
            env_step_keys,
            env_state,
            doer_action,
            fixed_positions=fixed_positions,
        )

        next_seer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done[:, None], jnp.zeros_like(x), x),
            next_seer_carry,
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done[:, None, None], jnp.zeros_like(x), x),
            next_doer_carry,
        )

        transition = TwoDoerTransition(
            global_obs=global_map,
            symbolic_obs=symbolic_obs,
            local_obs=local_obs,
            proprioception=proprioception,
            message=discrete_messages,
            target_images=env_obs["target_images"],
            menu_images=env_obs["menu_images"],
            pick_available=env_obs["pick_available"],
            doer_action=doer_action,
            doer_log_prob=doer_log_prob,
            value=value,
            reward=reward,
            task_reward=info["task_reward"],
            individual_selection_reward=info["individual_selection_reward"],
            valid_selection_count=info["valid_selection_count"],
            correct_selection_count=info["correct_selection_count"],
            eventual_success=info["eventual_success"],
            first_try_success=info["first_try_success"],
            progress_reward_per_doer=info["progress_reward_per_doer"],
            step_penalty_component=info["step_penalty"],
            wall_penalty_component=info["wall_penalty"],
            collision_penalty_component=info["collision_penalty"],
            done=done,
            advantage=jnp.zeros_like(reward),
            return_val=jnp.zeros_like(reward),
        )

        next_runner_state = (
            params,
            next_seer_carry,
            next_doer_carry,
            next_env_state,
            next_env_obs,
            rng,
            fixed_positions,
        )
        return next_runner_state, transition

    return rollout_step


@functools.partial(
    jax.jit,
    static_argnames=("num_steps", "step_fn", "critic_apply_fn"),
)
def generate_two_doer_trajectory_and_gae(
    params,
    rng,
    env_obs,
    env_state,
    seer_carry,
    doer_carry,
    fixed_positions,
    num_steps: int,
    step_fn,
    critic_apply_fn,
):
    initial_runner_state = (
        params,
        seer_carry,
        doer_carry,
        env_state,
        env_obs,
        rng,
        fixed_positions,
    )
    final_runner_state, trajectory_batch = jax.lax.scan(
        step_fn,
        initial_runner_state,
        None,
        length=num_steps,
    )
    _, _, _, _, final_env_obs, _, _ = final_runner_state
    last_val = critic_apply_fn(
        {"params": params["critic"]},
        final_env_obs["global_map"],
    ).squeeze(-1)
    advantages, returns = jax.vmap(
        compute_gae,
        in_axes=(1, 1, 1, 0, None, None),
        out_axes=1,
    )(
        trajectory_batch.reward,
        trajectory_batch.value,
        trajectory_batch.done,
        last_val,
        0.99,
        0.95,
    )
    trajectory_batch = trajectory_batch.replace(
        advantage=advantages,
        return_val=returns,
    )
    return final_runner_state, trajectory_batch


### `agents/mappo.py`

In [ ]:
%%writefile agents/mappo.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.training.train_state import TrainState
import optax
import chex
from typing import Tuple, Any, Callable
import distrax 
import optax
from training.action_masking import mask_pick_actions_until_menu_visible
from training.message_masking import hard_mask_inactive_message_bits

# 1. Data Structures: Meticulous shape tracking
# Using chex helps enforce accuracy by allowing us to assert shapes later.
@chex.dataclass
class Transition:
    global_obs: chex.Array  # For the Critic (CTDE)
    symbolic_obs: chex.Array # For the Seer
    local_obs: chex.Array   # For the Doer
    proprioception: chex.Array # For the Doer
    message: chex.Array     # For CIC and Heatmap logging
    target_images: chex.Array
    menu_images: chex.Array
    doer_action: chex.Array
    doer_log_prob: chex.Array
    seer_action: chex.Array
    seer_log_prob: chex.Array
    value: chex.Array
    reward: chex.Array
    task_reward: chex.Array
    progress_reward: chex.Array
    follow_reward: chex.Array
    cic_reward_component: chex.Array
    cic_score: chex.Array
    step_penalty_component: chex.Array
    bump_penalty_component: chex.Array
    done: chex.Array
    advantage: chex.Array
    return_val: chex.Array


@chex.dataclass
class TwoDoerTransition:
    global_obs: chex.Array
    symbolic_obs: chex.Array
    local_obs: chex.Array
    proprioception: chex.Array
    message: chex.Array
    target_images: chex.Array
    menu_images: chex.Array
    pick_available: chex.Array
    doer_action: chex.Array
    doer_log_prob: chex.Array
    value: chex.Array
    reward: chex.Array
    task_reward: chex.Array
    individual_selection_reward: chex.Array
    valid_selection_count: chex.Array
    correct_selection_count: chex.Array
    eventual_success: chex.Array
    first_try_success: chex.Array
    progress_reward_per_doer: chex.Array
    step_penalty_component: chex.Array
    wall_penalty_component: chex.Array
    collision_penalty_component: chex.Array
    done: chex.Array
    advantage: chex.Array
    return_val: chex.Array


def _build_message_codebook(message_levels: Tuple[int, ...], dtype: jnp.dtype) -> jnp.ndarray:
    axes = [jnp.arange(level, dtype=dtype) for level in message_levels]
    mesh = jnp.meshgrid(*axes, indexing="ij")
    return jnp.stack(mesh, axis=-1).reshape((-1, len(message_levels)))


def _compute_message_entropy_metrics(
    discrete_messages: chex.Array,
    message_levels: Tuple[int, ...],
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Approximate discrete-code usage with a soft histogram so gradients can flow."""
    flat_messages = discrete_messages.reshape((-1, discrete_messages.shape[-1]))
    codebook = _build_message_codebook(message_levels, flat_messages.dtype)
    sq_distances = jnp.sum(
        jnp.square(flat_messages[:, None, :] - codebook[None, :, :]),
        axis=-1,
    )
    assignment_probs = nn.softmax(-10.0 * sq_distances, axis=-1)
    code_probs = assignment_probs.mean(axis=0)
    code_probs = code_probs / (code_probs.sum() + 1e-8)
    message_entropy = -jnp.sum(code_probs * jnp.log(code_probs + 1e-8))

    num_codes = codebook.shape[0]
    if num_codes > 1:
        normalized_entropy = message_entropy / jnp.log(jnp.asarray(num_codes, dtype=flat_messages.dtype))
    else:
        normalized_entropy = jnp.asarray(0.0, dtype=flat_messages.dtype)

    dominant_code_prob = jnp.max(code_probs)
    return message_entropy, normalized_entropy, dominant_code_prob

# 2. Loss Functions: Separated from network definitions
def calculate_actor_losses(
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    actor_params: dict, # Changed from Any to dict
    transition_batch: Transition,
    init_seer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    control_mode: jnp.ndarray,
    message_levels: Tuple[int, ...],
    clip_eps: float = 0.2,
    entropy_coef: float = 0.01,
    seer_entropy_coef: jnp.ndarray = jnp.array(0.01)
) -> Tuple[jnp.ndarray, dict]:
    """Calculates separate PPO losses for the Seer and Doer reward streams."""
    
    # We must assert shape to prevent silent broadcasting bugs.
    # transition_batch has shape (batch_size, sequence_length, ...) or just (sequence_length, ...)
    # Wait, loop.py returns (num_steps, ...) for single env. Let's assume unbatched or reshape-able.
    # If the user is unrolling over dimension 0:
    
    def scan_fn(carry, transition_step):
        # Step shape assertions:
        # chex.assert_rank(transition_step.action, 1) # (batch_size,)
        # chex.assert_rank(transition_step.global_obs, 4) # (batch_size, H, W, C)
        
        seer_carry, doer_carry = carry
        
        # Seer Forward Pass
        next_seer_carry, discrete_message, thought_vector, seer_nav_logits = seer_apply_fn(
            {"params": actor_params["seer"]},
            seer_carry,
            transition_step.global_obs,
            transition_step.symbolic_obs,
            transition_step.target_images,
        )
        
        # Doer Forward Pass
        next_doer_carry, logits = doer_apply_fn(
            {"params": actor_params["doer"]},
            doer_carry,
            transition_step.local_obs,
            transition_step.proprioception,
            discrete_message,
            transition_step.menu_images
        )
        return (next_seer_carry, next_doer_carry), (
            logits,
            seer_nav_logits,
            discrete_message,
            thought_vector,
        )
        
    _, (doer_logits, seer_nav_logits, discrete_messages, thought_vectors) = jax.lax.scan(
        scan_fn, 
        (init_seer_carry, init_doer_carry), 
        transition_batch
    )

    communication_mode = control_mode == 1
    doer_pi = distrax.Categorical(logits=doer_logits)
    seer_pi = distrax.Categorical(logits=seer_nav_logits)
    doer_new_log_probs = doer_pi.log_prob(transition_batch.doer_action)
    seer_nav_new_log_probs = seer_pi.log_prob(transition_batch.seer_action)
    doer_entropy = doer_pi.entropy().mean()
    seer_nav_entropy = seer_pi.entropy().mean()

    seer_adv = transition_batch.advantage[..., 0]
    doer_adv = transition_batch.advantage[..., 1]
    seer_adv = (seer_adv - seer_adv.mean()) / (seer_adv.std() + 1e-8)
    doer_adv = (doer_adv - doer_adv.mean()) / (doer_adv.std() + 1e-8)

    seer_old_log_probs = jnp.where(
        communication_mode,
        transition_batch.doer_log_prob,
        transition_batch.seer_log_prob,
    )
    seer_new_log_probs = jnp.where(
        communication_mode,
        doer_new_log_probs,
        seer_nav_new_log_probs,
    )
    seer_logratio = seer_new_log_probs - seer_old_log_probs
    seer_ratio = jnp.exp(seer_logratio)

    seer_loss_unclipped = seer_ratio * seer_adv
    seer_loss_clipped = jnp.clip(seer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * seer_adv
    seer_actor_loss = -jnp.minimum(seer_loss_unclipped, seer_loss_clipped).mean()

    doer_logratio = doer_new_log_probs - transition_batch.doer_log_prob
    doer_ratio = jnp.exp(doer_logratio)
    doer_loss_unclipped = doer_ratio * doer_adv
    doer_loss_clipped = jnp.clip(doer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * doer_adv
    doer_actor_loss = -jnp.minimum(doer_loss_unclipped, doer_loss_clipped).mean()
    doer_actor_loss = jnp.where(communication_mode, doer_actor_loss, 0.0)

    active_discrete_messages = discrete_messages[..., :active_message_bits]
    message_entropy, message_entropy_normalized, dominant_code_prob = (
        _compute_message_entropy_metrics(active_discrete_messages, message_levels)
    )
    seer_bonus = jnp.where(communication_mode, message_entropy, seer_nav_entropy)

    seer_loss = seer_actor_loss - seer_entropy_coef * seer_bonus
    doer_loss = doer_actor_loss - jnp.where(communication_mode, entropy_coef * doer_entropy, 0.0)

    return (seer_loss, doer_loss), {
        "seer_actor_loss": seer_actor_loss,
        "doer_actor_loss": doer_actor_loss,
        "entropy": jnp.where(communication_mode, doer_entropy, seer_nav_entropy),
        "seer_ratio": seer_ratio.mean(),
        "doer_ratio": doer_ratio.mean(),
        "message_entropy": message_entropy,
        "message_entropy_normalized": message_entropy_normalized,
        "message_dominant_probability": dominant_code_prob,
        "seer_nav_entropy": seer_nav_entropy,
        "discrete_messages": discrete_messages
    }

def calculate_critic_loss(
    critic_apply_fn: Callable,
    critic_params: Any,
    transition_batch: Transition,
    value_clip: float = 0.2
) -> Tuple[jnp.ndarray, dict]:
    """Calculates the value loss for the centralized critic."""
    
    # Assert shape
    chex.assert_rank(transition_batch.global_obs, 4) # (batch_size, H, W, C) since it is flattened over sequence
    
    # The critic uses the global observation (CTDE)
    values = critic_apply_fn({"params": critic_params}, transition_batch.global_obs)

    value_pred_clipped = transition_batch.value + jnp.clip(
        values - transition_batch.value, -value_clip, value_clip
    )
    
    value_losses = jnp.square(values - transition_batch.return_val)
    value_losses_clipped = jnp.square(value_pred_clipped - transition_batch.return_val)
    
    critic_loss = 0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
    
    return critic_loss, {"critic_loss": critic_loss}


def calculate_two_doer_actor_losses(
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    actor_params: dict,
    transition_batch: TwoDoerTransition,
    init_seer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    message_levels: Tuple[int, ...],
    active_message_bits: int,
    pick_only_phase: bool,
    clip_eps: float = 0.2,
    entropy_coef: float = 0.01,
    seer_entropy_coef: jnp.ndarray = jnp.array(0.01),
) -> Tuple[jnp.ndarray, dict]:
    """PPO actor losses for one Seer coordinating two embodied Doers."""

    def scan_fn(carry, transition_step):
        seer_carry, doer_carry = carry
        next_seer_carry, discrete_messages, _, _ = seer_apply_fn(
            {"params": actor_params["seer"]},
            seer_carry,
            transition_step.global_obs,
            transition_step.symbolic_obs,
            transition_step.target_images,
        )
        discrete_messages = hard_mask_inactive_message_bits(
            discrete_messages,
            active_bits=active_message_bits,
        )
        batch_size, num_doers = transition_step.local_obs.shape[:2]
        flat_local_obs = transition_step.local_obs.reshape(
            (batch_size * num_doers,) + transition_step.local_obs.shape[2:]
        )
        flat_proprioception = transition_step.proprioception.reshape(
            (batch_size * num_doers,) + transition_step.proprioception.shape[2:]
        )
        flat_messages = discrete_messages.reshape(
            (batch_size * num_doers,) + discrete_messages.shape[2:]
        )
        flat_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_doer_carry, flat_logits = doer_apply_fn(
            {"params": actor_params["doer"]},
            flat_doer_carry,
            flat_local_obs,
            flat_proprioception,
            flat_messages,
            transition_step.menu_images.reshape((batch_size * num_doers,) + transition_step.menu_images.shape[2:])
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_doer_carry,
        )
        doer_logits = flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))
        doer_logits = mask_pick_actions_until_menu_visible(
            doer_logits,
            transition_step.menu_images,
            pick_only_phase=pick_only_phase,
            pick_available=transition_step.pick_available,
        )
        return (next_seer_carry, next_doer_carry), (doer_logits, discrete_messages)

    _, (doer_logits, discrete_messages) = jax.lax.scan(
        scan_fn,
        (init_seer_carry, init_doer_carry),
        transition_batch,
    )

    doer_pi = distrax.Categorical(logits=doer_logits)
    doer_new_log_probs = doer_pi.log_prob(transition_batch.doer_action)
    doer_entropy = doer_pi.entropy().mean()

    team_adv = transition_batch.advantage
    team_adv = (team_adv - team_adv.mean()) / (team_adv.std() + 1e-8)
    doer_old_log_probs = transition_batch.doer_log_prob

    seer_old_log_probs = doer_old_log_probs.sum(axis=-1)
    seer_new_log_probs = doer_new_log_probs.sum(axis=-1)
    seer_logratio = seer_new_log_probs - seer_old_log_probs
    seer_ratio = jnp.exp(seer_logratio)
    seer_loss_unclipped = seer_ratio * team_adv
    seer_loss_clipped = jnp.clip(seer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * team_adv
    seer_actor_loss = -jnp.minimum(seer_loss_unclipped, seer_loss_clipped).mean()

    doer_logratio = doer_new_log_probs - doer_old_log_probs
    doer_ratio = jnp.exp(doer_logratio)
    team_adv_expanded = team_adv[..., None]
    doer_loss_unclipped = doer_ratio * team_adv_expanded
    doer_loss_clipped = (
        jnp.clip(doer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * team_adv_expanded
    )
    doer_actor_loss = -jnp.minimum(doer_loss_unclipped, doer_loss_clipped).mean()

    active_discrete_messages = discrete_messages[..., :active_message_bits]
    message_entropy, message_entropy_normalized, dominant_code_prob = (
        _compute_message_entropy_metrics(active_discrete_messages, message_levels)
    )
    seer_loss = seer_actor_loss - seer_entropy_coef * message_entropy
    doer_loss = doer_actor_loss - entropy_coef * doer_entropy

    return (seer_loss, doer_loss), {
        "seer_actor_loss": seer_actor_loss,
        "doer_actor_loss": doer_actor_loss,
        "entropy": doer_entropy,
        "seer_ratio": seer_ratio.mean(),
        "doer_ratio": doer_ratio.mean(),
        "message_entropy": message_entropy,
        "message_entropy_normalized": message_entropy_normalized,
        "message_dominant_probability": dominant_code_prob,
        "discrete_messages": discrete_messages,
    }


def calculate_two_doer_critic_loss(
    critic_apply_fn: Callable,
    critic_params: Any,
    transition_batch: TwoDoerTransition,
    value_clip: float = 0.2,
) -> Tuple[jnp.ndarray, dict]:
    values = critic_apply_fn({"params": critic_params}, transition_batch.global_obs).squeeze(-1)
    value_pred_clipped = transition_batch.value + jnp.clip(
        values - transition_batch.value,
        -value_clip,
        value_clip,
    )
    value_losses = jnp.square(values - transition_batch.return_val)
    value_losses_clipped = jnp.square(value_pred_clipped - transition_batch.return_val)
    critic_loss = 0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
    return critic_loss, {"critic_loss": critic_loss}

# 3. The Update Step: JIT-compiled gradient application
import functools

@functools.partial(
    jax.jit,
    static_argnames=(
        "seer_apply_fn",
        "doer_apply_fn",
        "message_levels",
        "num_ppo_epochs",
        "num_minibatches",
    ),
)
def update_actor(
    actor_state: TrainState, 
    transition_batch: Transition,
    init_seer_carry: Any,
    init_doer_carry: Any,
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    control_mode: jnp.ndarray,
    message_levels: Tuple[int, ...],
    seer_entropy_coef: jnp.ndarray,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the actor network using PPO epochs."""
    
    # Add a batch dimension if missing (assumes trajectory is num_steps, ...)
    # Wait, the prompt says trajectory is (batch_size, seq_len, ...)
    # If the user scales num_envs later, batch_size = num_envs
    
    batch_size = transition_batch.doer_action.shape[0]
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        actor_state, key = carry
        key, subkey = jax.random.split(key)
        
        # Shuffle along the batch dimension
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            # Slice the minibatch
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], transition_batch)
            
            # Since calculate_actor_loss currently assumes scan over time, and time is dim 1:
            # wait, if input is (batch, time, ...) and scan is over time, we must swap axes!
            # scan_fn expects transition sequence to be the leading dimension.
            # So let's swap seq_len (dim 1) to be dim 0 for scan.
            mb_transition_time_first = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), mb_transition)
            
            # Extract initial carries for this minibatch
            # Assuming init_seer_carry is (batch_size, ...)
            mb_seer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_seer_carry)
            mb_doer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_doer_carry)
            
            def seer_loss_fn(seer_params):
                (seer_loss, _), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": seer_params, "doer": state.params["doer"]},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    control_mode=control_mode,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return seer_loss, metrics

            def doer_loss_fn(doer_params):
                (_, doer_loss), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": state.params["seer"], "doer": doer_params},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    control_mode=control_mode,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return doer_loss, metrics

            (seer_loss, seer_metrics), seer_grads = jax.value_and_grad(
                seer_loss_fn,
                has_aux=True,
            )(state.params["seer"])
            (doer_loss, doer_metrics), doer_grads = jax.value_and_grad(
                doer_loss_fn,
                has_aux=True,
            )(state.params["doer"])

            grads = {"seer": seer_grads, "doer": doer_grads}

            # Record explicit gradient norms for auditing
            seer_grad_norm = optax.global_norm(grads["seer"])
            doer_grad_norm = optax.global_norm(grads["doer"])

            metrics = {
                "seer_loss": seer_loss,
                "doer_loss": doer_loss,
                "seer_actor_loss": seer_metrics["seer_actor_loss"],
                "doer_actor_loss": doer_metrics["doer_actor_loss"],
                "entropy": doer_metrics["entropy"],
                "seer_ratio": seer_metrics["seer_ratio"],
                "doer_ratio": doer_metrics["doer_ratio"],
                "message_entropy": seer_metrics["message_entropy"],
                "message_entropy_normalized": seer_metrics["message_entropy_normalized"],
                "message_dominant_probability": seer_metrics["message_dominant_probability"],
                "seer_nav_entropy": seer_metrics["seer_nav_entropy"],
                "discrete_messages": seer_metrics["discrete_messages"],
                "seer_grad_norm": seer_grad_norm,
                "doer_grad_norm": doer_grad_norm,
                "actor_loss": seer_loss + doer_loss,
            }
            
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        # Minibatch loop (scan over start_indices)
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        actor_state, mb_metrics = jax.lax.scan(minibatch_fn, actor_state, start_indices)
        
        # Average metrics over minibatches
        epoch_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in mb_metrics.items()}
        return (actor_state, key), epoch_metrics
        
    (final_actor_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (actor_state, rng), None, length=num_ppo_epochs
    )
    
    # Return averaged metrics over epochs
    final_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in epoch_metrics.items()}
    return final_actor_state, final_metrics

@functools.partial(jax.jit, static_argnames=("critic_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_critic(
    critic_state: TrainState, 
    transition_batch: Transition,
    critic_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the critic network."""
    
    # For critic we can flatten the batch and sequence dimension since it's just MLPs
    batch_size = transition_batch.doer_action.shape[0] * transition_batch.doer_action.shape[1]
    flat_transition = jax.tree_util.tree_map(lambda x: x.reshape((batch_size,) + x.shape[2:]), transition_batch)
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        critic_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], flat_transition)
            
            loss_fn = lambda params: calculate_critic_loss(
                critic_apply_fn, params, mb_transition
            )
            
            (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        critic_state, mb_metrics = jax.lax.scan(minibatch_fn, critic_state, start_indices)
        
        epoch_metrics = jax.tree_util.tree_map(lambda x: x.mean(), mb_metrics)
        return (critic_state, key), epoch_metrics

    (final_critic_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (critic_state, rng), None, length=num_ppo_epochs
    )
    
    final_metrics = jax.tree_util.tree_map(lambda x: x.mean(), epoch_metrics)
    return final_critic_state, final_metrics


@functools.partial(
    jax.jit,
    static_argnames=(
        "seer_apply_fn",
        "doer_apply_fn",
        "message_levels",
        "active_message_bits",
        "pick_only_phase",
        "num_ppo_epochs",
        "num_minibatches",
    ),
)
def update_actor_two_doer(
    actor_state: TrainState,
    transition_batch: TwoDoerTransition,
    init_seer_carry: Any,
    init_doer_carry: Any,
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    message_levels: Tuple[int, ...],
    active_message_bits: int,
    pick_only_phase: bool,
    seer_entropy_coef: jnp.ndarray,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1,
) -> Tuple[TrainState, dict]:
    batch_size = transition_batch.doer_action.shape[0]
    minibatch_size = batch_size // num_minibatches

    def epoch_fn(carry, _):
        actor_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)

        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], transition_batch)
            mb_transition_time_first = jax.tree_util.tree_map(
                lambda x: jnp.swapaxes(x, 0, 1),
                mb_transition,
            )
            mb_seer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_seer_carry)
            mb_doer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_doer_carry)

            def seer_loss_fn(seer_params):
                (seer_loss, _), metrics = calculate_two_doer_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": seer_params, "doer": state.params["doer"]},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    message_levels=message_levels,
                    active_message_bits=active_message_bits,
                    pick_only_phase=pick_only_phase,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return seer_loss, metrics

            def doer_loss_fn(doer_params):
                (_, doer_loss), metrics = calculate_two_doer_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": state.params["seer"], "doer": doer_params},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    message_levels=message_levels,
                    active_message_bits=active_message_bits,
                    pick_only_phase=pick_only_phase,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return doer_loss, metrics

            (seer_loss, seer_metrics), seer_grads = jax.value_and_grad(
                seer_loss_fn,
                has_aux=True,
            )(state.params["seer"])
            (doer_loss, doer_metrics), doer_grads = jax.value_and_grad(
                doer_loss_fn,
                has_aux=True,
            )(state.params["doer"])
            grads = {"seer": seer_grads, "doer": doer_grads}
            seer_grad_norm = optax.global_norm(grads["seer"])
            doer_grad_norm = optax.global_norm(grads["doer"])
            metrics = {
                "seer_loss": seer_loss,
                "doer_loss": doer_loss,
                "seer_actor_loss": seer_metrics["seer_actor_loss"],
                "doer_actor_loss": doer_metrics["doer_actor_loss"],
                "entropy": doer_metrics["entropy"],
                "seer_ratio": seer_metrics["seer_ratio"],
                "doer_ratio": doer_metrics["doer_ratio"],
                "message_entropy": seer_metrics["message_entropy"],
                "message_entropy_normalized": seer_metrics["message_entropy_normalized"],
                "message_dominant_probability": seer_metrics["message_dominant_probability"],
                "discrete_messages": seer_metrics["discrete_messages"],
                "seer_grad_norm": seer_grad_norm,
                "doer_grad_norm": doer_grad_norm,
                "actor_loss": seer_loss + doer_loss,
            }
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics

        start_indices = jnp.arange(0, batch_size, minibatch_size)
        actor_state, mb_metrics = jax.lax.scan(minibatch_fn, actor_state, start_indices)
        epoch_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in mb_metrics.items()}
        return (actor_state, key), epoch_metrics

    (final_actor_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn,
        (actor_state, rng),
        None,
        length=num_ppo_epochs,
    )
    final_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in epoch_metrics.items()}
    return final_actor_state, final_metrics


@functools.partial(jax.jit, static_argnames=("critic_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_critic_two_doer(
    critic_state: TrainState,
    transition_batch: TwoDoerTransition,
    critic_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1,
) -> Tuple[TrainState, dict]:
    batch_size = transition_batch.doer_action.shape[0] * transition_batch.doer_action.shape[1]
    flat_transition = jax.tree_util.tree_map(
        lambda x: x.reshape((batch_size,) + x.shape[2:]),
        transition_batch,
    )
    minibatch_size = batch_size // num_minibatches

    def epoch_fn(carry, _):
        critic_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)

        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], flat_transition)
            loss_fn = lambda params: calculate_two_doer_critic_loss(
                critic_apply_fn,
                params,
                mb_transition,
            )
            (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics

        start_indices = jnp.arange(0, batch_size, minibatch_size)
        critic_state, mb_metrics = jax.lax.scan(minibatch_fn, critic_state, start_indices)
        epoch_metrics = jax.tree_util.tree_map(lambda x: x.mean(), mb_metrics)
        return (critic_state, key), epoch_metrics

    (final_critic_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn,
        (critic_state, rng),
        None,
        length=num_ppo_epochs,
    )
    final_metrics = jax.tree_util.tree_map(lambda x: x.mean(), epoch_metrics)
    return final_critic_state, final_metrics


### `eval/metrics.py`

In [ ]:
%%writefile eval/metrics.py
import jax
import jax.numpy as jnp
from typing import Tuple, Callable
import functools

@functools.partial(jax.jit, static_argnames=("doer_apply_fn",))
def compute_cic(
    doer_apply_fn: Callable,
    doer_params: dict,
    transition_batch, 
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    rng: jax.random.PRNGKey
) -> jnp.ndarray:
    """
    Computes Causal Influence of Communication (CIC) by ablating the Seer's message
    and measuring the divergence in the Doer's resulting deterministic actions.
    """
    
    def scan_fn(carry, step_data):
        doer_carry = carry
        msg, loc, prop, menu = step_data
        
        next_doer_carry, logits = doer_apply_fn(
            {"params": doer_params},
            doer_carry,
            loc,
            prop,
            msg,
            menu
        )
        return next_doer_carry, logits

    # Transition batch shapes: (seq_len, num_envs, ...) from loop.py
    loc = transition_batch.local_obs
    prop = transition_batch.proprioception
    msg_true = transition_batch.message
    menu = transition_batch.menu_images

    # True forward pass
    _, true_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_true, loc, prop, menu))
    
    # Ablated forward pass: shuffle messages independently over time for each env.
    env_keys = jax.random.split(rng, msg_true.shape[1])
    msg_shuffled = jax.vmap(
        lambda key, msgs: jax.random.permutation(key, msgs, axis=0)
    )(env_keys, jnp.swapaxes(msg_true, 0, 1))
    msg_shuffled = jnp.swapaxes(msg_shuffled, 0, 1)
    
    _, ablated_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_shuffled, loc, prop, menu))
    
    # Calculate CIC: divergence in deterministic policy
    true_actions = jnp.argmax(true_logits, axis=-1)
    ablated_actions = jnp.argmax(ablated_logits, axis=-1)
    
    cic = jnp.mean((true_actions != ablated_actions).astype(jnp.float32))
    
    return cic


@functools.partial(jax.jit, static_argnames=("doer_apply_fn",))
def compute_two_doer_cic(
    doer_apply_fn: Callable,
    doer_params: dict,
    transition_batch,
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    rng: jax.random.PRNGKey,
) -> jnp.ndarray:
    """
    Computes CIC for the two-Doer setting by ablating each private message stream
    and measuring how often the shared Doer policy changes its greedy action.
    """

    def scan_fn(carry, step_data):
        doer_carry = carry
        msg, loc, prop, menu = step_data
        batch_size, num_doers = loc.shape[:2]
        flat_loc = loc.reshape((batch_size * num_doers,) + loc.shape[2:])
        flat_prop = prop.reshape((batch_size * num_doers,) + prop.shape[2:])
        flat_msg = msg.reshape((batch_size * num_doers,) + msg.shape[2:])
        flat_menu = menu.reshape((batch_size * num_doers,) + menu.shape[2:])
        flat_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_carry, flat_logits = doer_apply_fn(
            {"params": doer_params},
            flat_carry,
            flat_loc,
            flat_prop,
            flat_msg,
            flat_menu,
        )
        next_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_carry,
        )
        logits = flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))
        return next_carry, logits

    loc = transition_batch.local_obs
    prop = transition_batch.proprioception
    msg_true = transition_batch.message
    menu = transition_batch.menu_images

    _, true_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_true, loc, prop, menu))

    flat_keys = jax.random.split(rng, msg_true.shape[1] * msg_true.shape[2])
    msg_shuffled = jnp.swapaxes(msg_true, 0, 1)
    msg_shuffled = jnp.swapaxes(msg_shuffled, 1, 2)
    msg_shuffled = msg_shuffled.reshape((-1, msg_true.shape[0], msg_true.shape[-1]))
    msg_shuffled = jax.vmap(
        lambda key, msgs: jax.random.permutation(key, msgs, axis=0)
    )(flat_keys, msg_shuffled)
    msg_shuffled = msg_shuffled.reshape(
        (msg_true.shape[1], msg_true.shape[2], msg_true.shape[0], msg_true.shape[-1])
    )
    msg_shuffled = jnp.swapaxes(msg_shuffled, 1, 2)
    msg_shuffled = jnp.swapaxes(msg_shuffled, 0, 1)

    _, ablated_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_shuffled, loc, prop, menu))

    true_actions = jnp.argmax(true_logits, axis=-1)
    ablated_actions = jnp.argmax(ablated_logits, axis=-1)
    cic = jnp.mean((true_actions != ablated_actions).astype(jnp.float32))
    return cic


## 3. Imports

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import jax
import jax.numpy as jnp
import optax
import flax.linen as nn
from flax.training.train_state import TrainState

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    wandb = None
    WANDB_AVAILABLE = False

from models.seer import Seer
from models.doer import Doer
from envs.two_doer_grid import TwoDoerBottleneckEnv, UNSET_TWO_DOER_POSITIONS
from training.loop import generate_two_doer_trajectory_and_gae, make_two_doer_rollout_step
from agents.mappo import update_actor_two_doer, update_critic_two_doer
from eval.metrics import compute_two_doer_cic

print(f"JAX backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")


## 4. Configuration

In [ ]:
config = {
    # Core
    "seed": 42,
    "use_wandb": False,           # Set True to log to Weights & Biases
    "wandb_project": "two-doers",
    "wandb_entity": None,

    # Architecture
    "fsq_levels": [2, 2, 2, 2],   # 4-bit FSQ -> 16 possible messages
    "hidden_size": 128,

    # Environment
    "grid_height": 10,
    "grid_width": 12,
    "local_view_size": 3,
    "corridor_length": 3,
    "episode_max_steps": 48,
    "doer_perception_level": 2,   # 1=local vision+coords, 2=no vision+coords, 3=blind

    # Rewards
    "goal_reward": 2.0,
    "progress_reward_scale": 0.1,
    "wrong_selection_penalty": 0.15,
    "wrong_selection_penalty_after_first": 0.30,
    "step_penalty": 0.03,
    "wall_penalty": 0.02,
    "collision_penalty": 0.05,

    # Curriculum
    "use_pick_object_curriculum": True,
    "two_doer_selection_level_start": 1,               # Start with pick-object phase
    "two_doer_selection_level_advance_threshold": 0.90, # Advance when success > 90%
    "two_doer_max_selection_attempts": 4,
    "pick_object_max_steps": 8,
    "pick_object_listen_steps": 1,

    # Training
    "learning_rate": 3e-4,
    "num_envs": 16,
    "num_steps": 64,
    "total_timesteps": 30_000_000,
    "seer_entropy_coef": 0.05,

    # Logging / Checkpointing
    "log_every": 10,
    "checkpoint_every": 1000,
    "checkpoint_dir": "checkpoints",
    "visualize_dir": "artifacts/episodes",
}

Path(config["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["visualize_dir"]).mkdir(parents=True, exist_ok=True)
print("Config ready.")


## 5. W&B (optional)

In [ ]:
if config["use_wandb"] and WANDB_AVAILABLE:
    wandb.init(project=config["wandb_project"], entity=config["wandb_entity"], config=config)
    print("W&B run:", wandb.run.url)
elif config["use_wandb"]:
    raise ImportError("wandb not installed. Run `pip install wandb` or set use_wandb=False.")
else:
    print("W&B disabled.")


## 6. Helpers

In [ ]:
class GlobalCritic(nn.Module):
    """CTDE critic: observes the full global map to produce a value baseline."""
    output_dim: int = 1

    @nn.compact
    def __call__(self, global_map: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=32, kernel_size=(3, 3))(global_map)
        x = nn.relu(x)
        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        return nn.Dense(features=self.output_dim)(x)


def initialize_two_doer_carry(doer, num_envs, num_doers, hidden_size):
    flat = doer.initialize_carry(batch_size=num_envs * num_doers, hidden_size=hidden_size)
    return jax.tree_util.tree_map(lambda x: x.reshape((num_envs, num_doers, hidden_size)), flat)


def get_active_message_levels(fsq_levels, active_bits):
    return tuple(fsq_levels[i] if i < active_bits else 1 for i in range(len(fsq_levels)))


def compute_message_stats(messages, active_levels):
    codes = np.asarray(messages).reshape(-1, messages.shape[-1])
    strs = [str(list(r)) for r in codes]
    counter = Counter(strs)
    total = len(strs)
    probs = np.array([c / total for c in counter.values()])
    entropy = float(-np.sum(probs * np.log(probs + 1e-8)))
    max_h = float(np.log(np.prod(active_levels)))
    return {
        "rollout_message_entropy_normalized": entropy / max_h if max_h > 0 else 0.0,
        "rollout_message_unique_codes": len(counter),
        "rollout_message_num_codes": int(np.prod(active_levels)),
    }


def save_checkpoint(params, step, label="checkpoint", checkpoint_dir="checkpoints"):
    from flax.training import checkpoints
    path = checkpoints.save_checkpoint(
        ckpt_dir=str(checkpoint_dir), target=params, step=int(step), overwrite=True,
    )
    print(f"[{label}] Saved -> {path}")
    return path


print("Helpers defined.")


## 7. Initialize Environment and Models

In [ ]:
rng = jax.random.PRNGKey(config["seed"])
rng, seer_rng, doer_rng, critic_rng = jax.random.split(rng, 4)

initial_phase = config["two_doer_selection_level_start"] if config["use_pick_object_curriculum"] else 2

env = TwoDoerBottleneckEnv(
    grid_height=config["grid_height"],
    grid_width=config["grid_width"],
    local_view_size=config["local_view_size"],
    corridor_length=config["corridor_length"],
    max_steps=config["episode_max_steps"],
    progress_reward_scale=config["progress_reward_scale"],
    goal_reward=config["goal_reward"],
    wrong_selection_penalty=config["wrong_selection_penalty"],
    wrong_selection_penalty_after_first=config["wrong_selection_penalty_after_first"],
    step_penalty=config["step_penalty"],
    wall_penalty=config["wall_penalty"],
    collision_penalty=config["collision_penalty"],
    doer_perception_level=config["doer_perception_level"],
    selection_phase_level=initial_phase,
    max_selection_attempts=config["two_doer_max_selection_attempts"],
    pick_object_max_steps=config["pick_object_max_steps"],
    pick_object_listen_steps=config["pick_object_listen_steps"],
)
print(f"Phase: {env.phase_name} (level={env.selection_phase_level})")
print(f"Doer perception level: {env.doer_perception_level}")
print(f"Actions: {env.num_actions} | Doers: {env.num_doers}")


In [ ]:
rng, reset_rng = jax.random.split(rng)
env_obs, env_state = env.reset_batch(jax.random.split(reset_rng, config["num_envs"]))

fsq_levels = config["fsq_levels"]
seer   = Seer(fsq_levels=fsq_levels, num_actions=env.num_actions, num_message_heads=env.num_doers)
doer   = Doer(fsq_levels=fsq_levels, num_actions=env.num_actions)
critic = GlobalCritic(output_dim=1)

dummy_map    = env_obs["global_map"][:1]
dummy_sym    = env_obs["symbolic_state"][:1]
dummy_local  = env_obs["local_views"][:1, 0]
dummy_prop   = env_obs["proprioceptions"][:1, 0]
dummy_target = env_obs["target_images"][:1]
dummy_menu   = env_obs["menu_images"][:1, 0]
dummy_msg    = jnp.zeros((1, len(fsq_levels)))
sc = seer.initialize_carry(1, config["hidden_size"])
dc = doer.initialize_carry(1, config["hidden_size"])

seer_params   = seer.init(seer_rng,   sc, dummy_map, dummy_sym, dummy_target)["params"]
doer_params   = doer.init(doer_rng,   dc, dummy_local, dummy_prop, dummy_msg, dummy_menu)["params"]
critic_params = critic.init(critic_rng, dummy_map)["params"]
params = {"seer": seer_params, "doer": doer_params, "critic": critic_params}

tx = optax.chain(optax.clip_by_global_norm(0.5), optax.adam(config["learning_rate"], eps=1e-5))
actor_state  = TrainState.create(apply_fn=None, params={"seer": seer_params, "doer": doer_params}, tx=tx)
critic_state = TrainState.create(apply_fn=critic.apply, params=critic_params, tx=tx)

seer_carry = seer.initialize_carry(config["num_envs"], config["hidden_size"])
doer_carry = initialize_two_doer_carry(doer, config["num_envs"], env.num_doers, config["hidden_size"])
step_fn    = make_two_doer_rollout_step(env, seer.apply, doer.apply, critic.apply)

num_updates = config["total_timesteps"] // (config["num_steps"] * config["num_envs"])
print(f"Models initialized. Total updates: {num_updates:,}")


## 8. Training Loop

Automatically transitions from Phase 1 (pick-object) to Phase 2 (full navigation + selection) once the success threshold is reached.

In [ ]:
phase1_checkpoint_saved = False

for update in range(num_updates):
    rng, rollout_rng = jax.random.split(rng)
    init_seer_carry = seer_carry
    init_doer_carry = doer_carry

    # Collect trajectory + compute GAE
    final_runner_state, trajectory_batch = generate_two_doer_trajectory_and_gae(
        params, rollout_rng, env_obs, env_state,
        seer_carry, doer_carry, UNSET_TWO_DOER_POSITIONS,
        config["num_steps"], step_fn, critic.apply,
    )
    params, seer_carry, doer_carry, env_state, env_obs, _, _ = final_runner_state

    # Update actor and critic
    rng, actor_rng, critic_rng_key = jax.random.split(rng, 3)
    batched = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), trajectory_batch)
    active_msg_levels = tuple(get_active_message_levels(fsq_levels, env.active_message_bits))

    actor_state, actor_metrics = update_actor_two_doer(
        actor_state, batched, init_seer_carry, init_doer_carry,
        seer.apply, doer.apply, actor_rng,
        active_msg_levels, env.active_message_bits, env.is_pick_object_phase,
        jnp.asarray(config["seer_entropy_coef"], dtype=jnp.float32),
    )
    critic_state, critic_metrics = update_critic_two_doer(
        critic_state, batched, critic.apply, critic_rng_key,
    )
    params["seer"]   = actor_state.params["seer"]
    params["doer"]   = actor_state.params["doer"]
    params["critic"] = critic_state.params

    # Metrics
    done_mask      = trajectory_batch.done
    n_episodes     = max(1, int(done_mask.sum()))
    first_try_rate = float(jnp.logical_and(done_mask, trajectory_batch.first_try_success).sum()) / n_episodes
    eventual_rate  = float(jnp.logical_and(done_mask, trajectory_batch.eventual_success).sum()) / n_episodes
    total_sel      = float(trajectory_batch.valid_selection_count.sum())
    correct_sel    = float(trajectory_batch.correct_selection_count.sum())
    sel_acc        = correct_sel / max(1.0, total_sel)
    msg_stats      = compute_message_stats(
        trajectory_batch.message[..., :env.active_message_bits], active_msg_levels
    )

    # Logging
    if update % config["log_every"] == 0:
        global_step = (update + 1) * config["num_steps"] * config["num_envs"]
        print(
            f"[{update:5d}/{num_updates}] Phase={env.phase_name:20s} | "
            f"Success={first_try_rate:.3f} | Eventual={eventual_rate:.3f} | "
            f"SelAcc={sel_acc:.3f} | "
            f"MsgH={msg_stats['rollout_message_entropy_normalized']:.3f} | "
            f"MsgUsed={msg_stats['rollout_message_unique_codes']}/{msg_stats['rollout_message_num_codes']} | "
            f"DoerGrad={actor_metrics.get('doer_grad_norm', 0.0):.4f}"
        )
        if config["use_wandb"] and WANDB_AVAILABLE:
            wandb.log({
                "phase": env.selection_phase_level,
                "first_try_success_rate": first_try_rate,
                "eventual_success_rate": eventual_rate,
                "selection_accuracy": sel_acc,
                "message_entropy_normalized": msg_stats["rollout_message_entropy_normalized"],
                "message_unique_codes": msg_stats["rollout_message_unique_codes"],
                "team_reward": float(trajectory_batch.reward.mean()),
                "seer_grad_norm": actor_metrics.get("seer_grad_norm", 0.0),
                "doer_grad_norm": actor_metrics.get("doer_grad_norm", 0.0),
                "critic_loss": critic_metrics.get("critic_loss", 0.0),
            }, step=global_step)

    # Phase 1 -> Phase 2 transition
    if (
        config["use_pick_object_curriculum"]
        and env.selection_phase_level == 1
        and first_try_rate > config["two_doer_selection_level_advance_threshold"]
    ):
        global_step = (update + 1) * config["num_steps"] * config["num_envs"]
        print("\n" + "=" * 60)
        print("  PHASE TRANSITION: Pick-Object -> Full Training")
        print(f"  Success={first_try_rate:.3f} > threshold={config['two_doer_selection_level_advance_threshold']}")
        print("=" * 60 + "\n")

        if not phase1_checkpoint_saved:
            save_checkpoint(params, global_step, label="phase1_complete", checkpoint_dir=config["checkpoint_dir"])
            phase1_checkpoint_saved = True

        env.set_selection_phase_level(2)
        step_fn = make_two_doer_rollout_step(env, seer.apply, doer.apply, critic.apply)

        rng, reset_rng = jax.random.split(rng)
        env_obs, env_state = env.reset_batch(jax.random.split(reset_rng, config["num_envs"]))
        seer_carry = seer.initialize_carry(config["num_envs"], config["hidden_size"])
        doer_carry = initialize_two_doer_carry(doer, config["num_envs"], env.num_doers, config["hidden_size"])

    # Periodic checkpoint
    if update > 0 and update % config["checkpoint_every"] == 0:
        global_step = (update + 1) * config["num_steps"] * config["num_envs"]
        save_checkpoint(params, global_step, label=f"update_{update}", checkpoint_dir=config["checkpoint_dir"])

print("Training complete.")


## 9. Save Final Checkpoint

In [ ]:
final_step = num_updates * config["num_steps"] * config["num_envs"]
save_checkpoint(params, final_step, label="final", checkpoint_dir=config["checkpoint_dir"])
if config["use_wandb"] and WANDB_AVAILABLE:
    wandb.finish()
